---
title: "DRG Cleaning"

author: "Carlos Resurreccion"

date: "2024-07-01"

---

In [19]:
knitr::opts_chunk$set(echo = TRUE)


## Load Required Libraries

In [20]:
options(verbose = FALSE)
options(warn = -1)


In [21]:
library(here)
source(here("data-cleaning", "r_scripts", "libraries.R"))


In [22]:
options(warn = 1)


## Set Parameters

### Year to Load, Version, and Parameters

In [23]:
source(here("data-cleaning", "r_scripts", "parameters.R"))


## Source Data Formats

In [24]:
source(here("data-cleaning", "r_scripts", "data-formats.R"))
source(here("data-cleaning", "r_scripts", "file-paths.R"))


## Source Functions

In [25]:
source(here("data-cleaning", "r_scripts", "general-functions.R"))
source(here("data-cleaning", "r_scripts", "main-functions.R"))
source(here("data-cleaning", "r_scripts", "icd-functions.R"))
source(here("data-cleaning", "r_scripts", "rvs-functions.R"))
source(here("data-cleaning", "r_scripts", "pdx-functions.R"))
source(here("data-cleaning", "r_scripts", "grouper-functions.R"))


## Load Mapping Data

In [26]:
proc <- fread(here(path_to_excel, "proc.csv"))
proc[, CODE := as.character(CODE)]

rvs_icd9 <- fread(here(path_to_aux, "rvs_icd9cm.csv"),
  select = c("rvs", "icd9cm")
)
rvs_icd9[, rvs := as.character(rvs)]
rvs_icd9[, icd9cm := as.character(icd9cm * 100)]
rvs_icd9 <- merge(rvs_icd9, proc[, .(CODE, DRGUSE)],
  by.x = "icd9cm", by.y = "CODE", all.x = TRUE
)
rvs_icd9[, is_drg := !is.na(DRGUSE) & DRGUSE]
rvs_icd9 <- rvs_icd9[!is.na(rvs) & !is.na(icd9cm), -"DRGUSE"]

acr_rvs <- fread(here(path_to_aux, "acr_rvs.csv"))

# Read in the data.table
tdrg_icd10 <- fread(here(path_to_aux, "i10.csv"))

# Set the key if not already set
setkey(tdrg_icd10, "CODE")

# Subset and assign the result to acc_pdx
acc_pdx <- tdrg_icd10[ACCPDX == "Y", CODE]

# Optional: if CODEs are not unique in tdrg_icd10
acc_pdx <- unique(acc_pdx)


## Read Data

### Reading Data

In [27]:
options(verbose = FALSE)
options(warn = -1)

dt <- main_read_function()

if (file.exists(total_rows_file)) {
  total_rows <- readRDS(total_rows_file)
} else {
  total_rows <- fread(full_claims, select = 1L, header = TRUE)[, .N]
  saveRDS(total_rows, file = total_rows_file)
}


[1] "Sampled file exists. Reading the sampled file..."
[1] "Sampled file matches sample size."


In [28]:
options(warn = 1)


## Data Processing

### Data Cleaning

#### Not chunking

In [29]:
if (!to_chunk) {
  tic("Total execution time:")
  if (to_profvis) {
    profvis({
      dt <- clean_data(dt)
    })
  } else {
    dt <- clean_data(dt)
  }
}


#### Chunking

In [30]:
if (to_chunk) {
  tic("Total execution time:")
  if (to_profvis) {
    profvis({
      num_cores <- max(1, availableCores() - 1)
      chunk_size <- ceiling(nrow(dt) / num_cores)
      chunks <- split(dt, rep(1:num_cores,
        each = chunk_size,
        length.out = nrow(dt)
      ))
      # Plan for parallel processing
      plan(multisession, workers = num_cores)
      # Process each chunk in parallel
      processed_chunks <- future_lapply(chunks, process_chunk,
        future.seed = global_seed
      )
      # Combine processed chunks
      dt <- rbindlist(processed_chunks)
      dt <- replace_empty_with_na(dt, to_view_checks)
    })
  } else {
    num_cores <- max(1, availableCores() - 1)
    chunk_size <- ceiling(nrow(dt) / num_cores)
    chunks <- split(dt, rep(1:num_cores,
      each = chunk_size,
      length.out = nrow(dt)
    ))
    # Plan for parallel processing
    plan(multisession, workers = num_cores)
    # Process each chunk in parallel
    processed_chunks <- future_lapply(chunks, process_chunk,
      future.seed = global_seed
    )
    # Combine processed chunks
    dt <- rbindlist(processed_chunks)
    dt <- replace_empty_with_na(dt, to_view_checks)
  }
}


[1] "Viewing checks"
 [1] "id_series"           "id_pin"              "date_adm"           
 [4] "time_adm"            "date_dis"            "time_dis"           
 [7] "date_rec"            "date_ref"            "date_check"         
[10] "id_hci"              "id_hcp"              "clin_outpatient"    
[13] "clin_emergency"      "pat_type"            "clin_acc"           
[16] "pat_rel"             "pat_bdate"           "pat_age"            
[19] "pat_sex"             "pat_bwt"             "pat_memcat_parent"  
[22] "pat_memcat_child"    "pat_memcat_subchild" "clin_discharge"     
[25] "clin_c1"             "clin_c2"             "clin_icd1"          
[28] "clin_icd2"           "clin_icd3"           "clin_icd4"          
[31] "clin_icd5"           "clin_icd6"           "clin_icd7"          
[34] "clin_icd8"           "clin_icd9"           "clin_icd10"         
[37] "clin_icd11"          "clin_icd12"          "clin_rvs1"          
[40] "clin_rvs2"           "clin_rvs3"           "clin_r

Warning message in append_and_remove_rvs(dt$clin_rvs, dt$clin_c1, rvs_icd9):
"Discarded RVS codes: 77401"


[1] "Viewing checks"
 [1] "id_series"           "id_pin"              "date_adm"           
 [4] "time_adm"            "date_dis"            "time_dis"           
 [7] "date_rec"            "date_ref"            "date_check"         
[10] "id_hci"              "id_hcp"              "clin_outpatient"    
[13] "clin_emergency"      "pat_type"            "clin_acc"           
[16] "pat_rel"             "pat_bdate"           "pat_age"            
[19] "pat_sex"             "pat_bwt"             "pat_memcat_parent"  
[22] "pat_memcat_child"    "pat_memcat_subchild" "clin_discharge"     
[25] "clin_c1"             "clin_c2"             "clin_icd1"          
[28] "clin_icd2"           "clin_icd3"           "clin_icd4"          
[31] "clin_icd5"           "clin_icd6"           "clin_icd7"          
[34] "clin_icd8"           "clin_icd9"           "clin_icd10"         
[37] "clin_icd11"          "clin_icd12"          "clin_rvs1"          
[40] "clin_rvs2"           "clin_rvs3"           "clin_r

### Map codes and Find PDx (if not chunking)

#### Map Codes

In [31]:
# Map RVS codes
if (!to_chunk) {
  if (to_profvis) {
    profvis({
      dt[, icd9_list := map_rvs_icd9(clin_rvs, rvs_icd9)]
      clin_c1 <- dt$clin_c1
      clin_c2 <- dt$clin_c2
      clin_icd <- dt$clin_icd

      # Apply the ICD-10 mappings
      mapped_columns <- implement_icd10_mapping(
        clin_c1, clin_c2, clin_icd, tdrg_icd10
      )

      # Save the results back to the data.table
      dt[, clin_c1 := mapped_columns$clin_c1]
      dt[, clin_c2 := mapped_columns$clin_c2]
      dt[, clin_icd := mapped_columns$clin_icd]
      # Replace empty strings in character and factor columns with NA
      # dt <- replace_empty_with_na(dt, to_view_checks)
    })
  } else {
    dt[, icd9_list := map_rvs_icd9(clin_rvs, rvs_icd9)]
    # Extract specific columns
    clin_c1 <- dt$clin_c1
    clin_c2 <- dt$clin_c2
    clin_icd <- dt$clin_icd

    # Apply the ICD-10 mappings
    mapped_columns <- implement_icd10_mapping(
      clin_c1, clin_c2, clin_icd, tdrg_icd10
    )

    # Save the results back to the data.table
    dt[, clin_c1 := mapped_columns$clin_c1]
    dt[, clin_c2 := mapped_columns$clin_c2]
    dt[, clin_icd := mapped_columns$clin_icd]
    # Replace empty strings in character and factor columns with NA
    # dt <- replace_empty_with_na(dt, to_view_checks)
  }
}


#### Find PDx

In [32]:
# Map RVS codes
if (!to_chunk) {
  if (to_profvis) {
    profvis({
      # dt <- apply_find_pdx(dt)
      pdx_result <- apply_find_pdx(dt$clin_c1, dt$clin_c2, dt$clin_icd)
      dt$pdx <- pdx_result$pdx
      dt$pdx_code <- pdx_result$pdx_code
    })
  } else {
    # dt <- apply_find_pdx(dt)
    pdx_result <- apply_find_pdx(dt$clin_c1, dt$clin_c2, dt$clin_icd)
    dt$pdx <- pdx_result$pdx
    dt$pdx_code <- pdx_result$pdx_code
  }
}


In [33]:
if (to_write) {
  fwrite(dt, here(path_to_intermediate, paste0(
    "output_", year_to_load,
    suffix, ".csv"
  )))
}


## Export

### Export for Batch Grouper

In [34]:
if (to_group) {
  if (to_profvis) {
    profvis({
      export_for_batch_grouper(dt, year_to_load, output_txt_file)
      for_batch_grouping <- fread(output_txt_file,
        sep = "|", na.strings = "--"
      )
      batch_grouping_result <- fread(grouper_result_file,
        sep = "|", na.strings = "--"
      )
    })
  } else {
    export_for_batch_grouper(dt, year_to_load, output_txt_file)
    for_batch_grouping <- fread(output_txt_file,
      sep = "|", na.strings = "--"
    )
    batch_grouping_result <- fread(grouper_result_file,
      sep = "|", na.strings = "--"
    )
  }
}


## Runtime Estimation

### Stop Timer

In [35]:
# Stop the timer and capture total time
toc_data <- toc(log = TRUE)
total_time <- toc_data$toc - toc_data$tic


Total execution time:: 6.02 sec elapsed


### Calculate Speed

In [36]:
# Calculate time spent per cell and per row
total_rows_dt <- nrow(dt)
total_cells <- nrow(dt) * ncol(dt)

time_per_cell <- total_time / total_cells
time_per_row <- total_time / total_rows_dt
time_estimate_total_rows <- time_per_row * total_rows

# Format the row numbers
formatted_total_rows_dt <- format_large_numbers(total_rows_dt)
formatted_total_rows <- format_large_numbers(total_rows)

# Print the results with aligned decimal points and formatted row numbers
cat(sprintf(
  "Time spent (total) for %2s rows:  %1.2f sec  (actual)\n",
  formatted_total_rows_dt, total_time
))
cat(sprintf(
  "Time spent (t/row) for %2s rows:  %1.2f msec (actual)\n",
  formatted_total_rows_dt, time_per_row * 1000
))
cat(sprintf(
  "Time spent (total) for  %2s rows: %2.2f min  (estimate)\n",
  formatted_total_rows, time_estimate_total_rows / 60
))


Time spent (total) for 25.0k rows:  6.02 sec  (actual)
Time spent (t/row) for 25.0k rows:  0.24 msec (actual)
Time spent (total) for  11.8m rows: 47.27 min  (estimate)
